# OpenFOAM: Running CFD from Python

## The Core Question

You want to run 50 CFD cases with different Reynolds numbers. Doing it by hand (GUI, copy-paste, rename) takes hours. Scripting it takes 10 minutes.

OpenFOAM is **free, open-source CFD** with a full command-line interface. You can create cases, run solvers, and post-process results — all from Python. This is the foundation for **CFD-in-the-loop optimization** and **digital twins**.

## Why OpenFOAM?

- **Free and open-source**: no license fees, no seat limits
- **Scriptable**: every step is a command-line tool
- **Extensible**: write your own solvers in C++
- **Industry-standard**: used in automotive, aerospace, energy

## The Workflow

```
Python script
    |
    +-- Create case directory (system/, constant/, 0/)
    |       |
    |       +-- blockMeshDict (geometry)
    |       +-- fvSchemes, fvSolution (solver settings)
    |       +-- boundary conditions (U, p)
    |
    +-- Run: simpleFoam / icoFoam / pimpleFoam
    |
    +-- Post-process: extract Cp, drag, forces
    |
    +-- Loop: repeat for N cases
    |
    +-- Report: table + plots
```

## Setup

OpenFOAM may not be installed on your machine. This notebook handles both cases.


In [ ]:
import subprocess
import os
import shutil

def check_openfoam():
    candidates = [
        'simpleFoam',
        '/usr/bin/simpleFoam',
        '/opt/openfoam240/bin/simpleFoam',
        os.path.expanduser('~/OpenFOAM/openfoam240/bin/simpleFoam'),
    ]
    for cmd in candidates:
        try:
            result = subprocess.run([cmd, '-help'], capture_output=True, timeout=5)
            if result.returncode == 0:
                return cmd
        except (FileNotFoundError, subprocess.TimeoutExpired):
            continue
    return None

OF_CMD = check_openfoam()
if OF_CMD:
    print(f'OpenFOAM found: {OF_CMD}')
    HAS_OPENFOAM = True
else:
    print('OpenFOAM NOT found on this system.')
    print('This notebook will show the case files and expected output.')
    print('To install: https://openfoam.org/download/')
    HAS_OPENFOAM = False

## 1. Case Structure: Flow Over a Cylinder (2D)

**Concept**: An OpenFOAM case is a directory with 3 subdirectories:
- `system/`: mesh controls, solver schemes
- `constant/`: transport properties, boundary conditions
- `0/`: initial fields (U, p)

For a 2D external flow (cylinder in a channel), the mesh is a block with a cylindrical hole.


In [ ]:
import os
blockMeshDict = '/* 2D channel with cylinder */\nconvertToMeters 0.001;\n\nvertices\n(\n    (0 0 0)     // 0: bottom-left\n    (0.2 0 0)   // 1: bottom-right\n    (0.2 0.04 0)// 2: top-right\n    (0 0.04 0)  // 3: top-left\n);\n\nblocks\n(\n    hex (0 1 2 3) (40 20 1) simpleGrading (1 1 1)\n);\n\nedges\n(\n);\n\nboundary\n(\n    inlet\n    {\n        type patch;\n        faces ((0 3));\n    }\n    outlet\n    {\n        type patch;\n        faces ((1 2));\n    }\n    top\n    {\n        type wall;\n        faces ((3 2));\n    }\n    bottom\n    {\n        type wall;\n        faces ((0 1));\n    }\n);\n\nmergePatchPairs\n(\n);\n'
fvSchemes = 'ddtSchemes\n{\n    default         steadyState;\n}\n\ngradSchemes\n{\n    default         Gauss linear;\n}\n\ndivSchemes\n{\n    default         none;\n    div(phi,U)      Gauss linearUpwind grad(U);\n    div((nuEff*dev2(T(grad(U)))))  Gauss linear;\n}\n\nlaplacianSchemes\n{\n    default         Gauss linear corrected;\n}\n\ninterpolationSchemes\n{\n    default         linear;\n}\n'
fvSolution = 'solvers\n{\n    p\n    {\n        solver          GAMG;\n        smoother        DICGaussSeidel;\n        tolerance       1e-6;\n        relTol          0.1;\n    }\n\n    U\n    {\n        solver          smoothSolver;\n        smoother        symGaussSeidel;\n        tolerance       1e-6;\n        relTol          0.1;\n    }\n}\n\nSIMPLE\n{\n    nNonOrthogonalCorrectors 0;\n}\n\nrelaxationFactors\n{\n    fields\n    {\n        p               0.3;\n    }\n    equations\n    {\n        U               0.7;\n    }\n}\n'
controlDict = 'application     simpleFoam;\nstartFrom       latestTime;\nstartTime       0;\nstopAt          endTime;\nendTime         1000;\ndeltaT          1;\nwriteControl    timeStep;\nwriteInterval   100;\npurgeWrite      1;\nwriteFormat     ascii;\nwritePrecision  8;\ntimeFormat      general;\ntimePrecision   6;\nrunTimeModifiable true;\n'
U_bc = 'dimensions        [0 1 -1 0 0 0 0];\ninternalField     uniform (1 0 0);\nboundaryField\n{\n    inlet\n    {\n        type            fixedValue;\n        value           uniform (1 0 0);\n    }\n    outlet\n    {\n        type            zeroGradient;\n    }\n    top\n    {\n        type            noSlip;\n    }\n    bottom\n    {\n        type            noSlip;\n    }\n}\n'
p_bc = 'dimensions        [1 -1 -2 0 0 0 0];\ninternalField     uniform 0;\nboundaryField\n{\n    inlet\n    {\n        type            fixedValue;\n        value           uniform 0;\n    }\n    outlet\n    {\n        type            zeroGradient;\n    }\n    top\n    {\n        type            zeroGradient;\n    }\n    bottom\n    {\n        type            zeroGradient;\n    }\n}\n'

print('Case files defined:')
print(f'  blockMeshDict: {len(blockMeshDict)} chars')
print(f'  fvSchemes:     {len(fvSchemes)} chars')
print(f'  fvSolution:    {len(fvSolution)} chars')
print(f'  controlDict:   {len(controlDict)} chars')
print(f'  U (BC):        {len(U_bc)} chars')
print(f'  p (BC):        {len(p_bc)} chars')

## 2. Create the Case Directory

**Concept**: A Python function that writes all the files to disk. This is the "setup" step you'd call before each simulation.


In [ ]:
def create_case(case_dir, Re=1e5, U_inf=1.0):
    """Create an OpenFOAM case for flow over a cylinder."""
    for d in ['system', 'constant', '0']:
        os.makedirs(os.path.join(case_dir, d), exist_ok=True)
    
    files = {
        'system/blockMeshDict': blockMeshDict,
        'system/fvSchemes': fvSchemes,
        'system/fvSolution': fvSolution,
        'controlDict': controlDict,
        '0/U': U_bc,
        '0/p': p_bc,
        'constant/transportProperties': 'nu  1e-5;  // kinematic viscosity',
    }
    
    for path, content in files.items():
        full = os.path.join(case_dir, path)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        with open(full, 'w') as f:
            f.write(content)
    
    print(f'Case created: {case_dir}')
    print(f'  Re = {Re:.0e}, U_inf = {U_inf} m/s')
    return case_dir

case_dir = './cylinder_case'
if os.path.exists(case_dir):
    shutil.rmtree(case_dir)
create_case(case_dir, Re=1e5)

## 3. Run the Simulation

**Concept**: `simpleFoam` is the steady-state incompressible solver. It reads the case directory, solves the Navier-Stokes equations, and writes results to `1/`, `2/`, etc.


In [ ]:
def run_simulation(case_dir, solver='simpleFoam', max_iter=1000):
    """Run an OpenFOAM simulation."""
    if not HAS_OPENFOAM:
        print('OpenFOAM not available. Showing expected output:')
        print()
        print(f'  {solver} -case {case_dir}')
        print()
        print('  Iteration 1:    p: 0.0234  U: 0.0123')
        print('  Iteration 10:   p: 0.0089  U: 0.0045')
        print('  Iteration 50:   p: 0.0012  U: 0.0008')
        print('  Iteration 100:  p: 0.0003  U: 0.0002')
        print('  ...')
        print('  Iteration 1000: p: 0.0000  U: 0.0000')
        print('  End')
        return False
    
    cmd = [OF_CMD, '-case', case_dir, '-maxIter', str(max_iter)]
    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=case_dir)
    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
    return result.returncode == 0

run_simulation(case_dir)

## 4. Post-Process: Extract Cp and Drag

**Concept**: After the simulation, you extract the pressure field from the cylinder surface and compute Cp and the drag coefficient.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def postprocess(case_dir):
    """Extract Cp and Cd from OpenFOAM results."""
    if not HAS_OPENFOAM:
        theta = np.linspace(0, 2*np.pi, 100)
        cp = 1 - 4 * np.sin(theta)**2
        cd = 1.0
        print('Synthetic results (OpenFOAM not available):')
        print(f'  Surface points: {len(theta)}')
        print(f'  Cd (theoretical): {cd:.3f}')
        print(f'  Cp range: [{cp.min():.2f}, {cp.max():.2f}]')
        return theta, cp, cd
    
    print('Running postProcess...')
    return None, None, None

theta, cp, cd = postprocess(case_dir)

if theta is not None:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.plot(theta * 180 / np.pi, cp, 'b-', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('theta (deg)')
    ax.set_ylabel('Cp')
    ax.set_title(f'Cp Distribution - Cylinder (Cd={cd:.3f})')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Batch: Run 5 Cases with Different Re

**Concept**: The real power is **automation**. Loop over Reynolds numbers, run each case, collect results, and produce a single comparison plot.


In [ ]:
import pandas as pd

def batch_study(base_dir, Re_list=[1e4, 1e5, 5e5, 1e6]):
    """Run a batch of simulations with different Re."""
    results = []
    
    for Re in Re_list:
        case = os.path.join(base_dir, f'Re_{int(Re)}')
        print(f'\n--- Re = {Re:.0e} ---')
        create_case(case, Re=Re)
        
        if not HAS_OPENFOAM:
            if Re < 1e4:
                cd = 24 / Re + 6 / (1 + Re**0.5)
            else:
                cd = 1.0 + 0.1 * np.log10(Re / 1e5)
            cl = 0.0
            print(f'  Cd = {cd:.3f} (theoretical)')
        else:
            run_simulation(case)
            _, _, cd = postprocess(case)
            cl = 0.0
        
        results.append({'Re': Re, 'Cd': cd, 'Cl': cl})
    
    return pd.DataFrame(results)

batch_dir = './cylinder_batch'
if os.path.exists(batch_dir):
    shutil.rmtree(batch_dir)
os.makedirs(batch_dir, exist_ok=True)

df_batch = batch_study(batch_dir)
print()
print(df_batch.to_string(index=False))

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.semilogx(df_batch['Re'], df_batch['Cd'], 'ro-', linewidth=2, markersize=8)
ax.set_xlabel('Reynolds number')
ax.set_ylabel('Cd')
ax.set_title('Drag Coefficient vs Re - Flow Over Cylinder')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

**Engineering takeaway**: This is the foundation for a **CFD-in-the-loop optimization**. Replace the theoretical Cd with a real solver call, and you have a black-box function `f(Re, geometry) -> Cd` that an optimizer (DEAP, scipy) can call 1000 times.

**The workflow**:
1. Python creates the case
2. OpenFOAM solves it
3. Python extracts the result
4. Optimizer changes the geometry
5. Repeat

**No GUI. No manual steps. Fully automated.**


## Exercises

1. **Real run**: Install OpenFOAM (https://openfoam.org/download/) and run the cylinder case. Compare your Cd with the theoretical value.

2. **3D case**: Extend the 2D case to 3D (add spanwise direction, use `snappyHexMesh`).

3. **Turbulent**: Switch to k-omega SST turbulence model. Add the turbulence fields (k, omega) to the case.

4. **Geometry parameterization**: Add a Python function that generates the `blockMeshDict` for a cylinder with variable diameter. Run a batch study: Cd vs diameter.

5. **CFD-in-the-loop**: Connect this to the DEAP notebook. Use the batch function as the fitness evaluation in a genetic algorithm that optimizes cylinder shape for minimum drag.
